# Example 2.4 / 1: Loading a Labeled Frame

Companion notebook for **2.4 Data from pandas, NumPy and Files** in the AMPL lab Documentation, section "Loading a Labeled Frame". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Loading a Labeled Frame

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                  # Products: the DataFrame index
param rate {PROD} > 0;     # Production rate in tons per hour
param profit {PROD};       # Profit per ton
param market {PROD} >= 0;  # Maximum sales in tons

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

products = pd.DataFrame(
    {
        "rate": [200, 140, 160],
        "profit": [25, 30, 29],
        "market": [6000, 4000, 3500],
    },
    index=pd.Index(["bands", "coils", "plate"], name="PROD"),
)
ampl.set_data(products, "PROD")

In [5]:
# run.py
print(ampl.param["profit"].to_pandas())

       profit
bands      25
coils      30
plate      29


### Pure AMPL: data.dat and model.run

The same labeled rows in a data file:

In [6]:
%%writefile data.dat
param: PROD: rate profit market :=
  bands 200 25 6000
  coils 140 30 4000
  plate 160 29 3500;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display profit;

profit [*] :=
bands  25
coils  30
plate  29
;

